In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [14]:
import os

print("CWD:", os.getcwd())
print("data:", os.listdir("../data"))
print("raw:", os.listdir("../data/raw"))

CWD: c:\Users\ojgue\OneDrive\Documentos\FORECASTINGVENTAS\notebooks
data: ['processed', 'raw', 'raw copy']
raw: ['.gitkeep', 'entrenamiento', 'inferencia']


In [15]:
import os

ruta_inferencia = None

for root, dirs, files in os.walk(".."):
    if "ventas_2025_inferencia.csv" in files:
        ruta_inferencia = os.path.join(root, "ventas_2025_inferencia.csv")
        print("Archivo encontrado en:", ruta_inferencia)

if ruta_inferencia is None:
    print("❌ No se encontró el archivo")
else:
    inferencia_df = pd.read_csv(ruta_inferencia)
    print(inferencia_df.head())

Archivo encontrado en: ..\data\raw\inferencia\ventas_2025_inferencia.csv
        fecha producto_id                            nombre categoria  \
0  2025-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1  2025-10-25    PROD_002              Adidas Ultraboost 23   Running   
2  2025-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3  2025-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4  2025-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True               26.0   
1  Zapatillas Running          135         True               27.0   
2  Zapatillas Running           85        False                5.0   
3  Zapatillas Running           75        False                3.0   
4        Ropa Running           35        False                3.0   

   precio_venta  ingresos  Amazon  Decathlon  Deporvillage  
0     

In [16]:
ruta_inferencia = "../data/raw/inferencia/ventas_2025_inferencia.csv"

inferencia_df = pd.read_csv(ruta_inferencia)

print(inferencia_df.head())

        fecha producto_id                            nombre categoria  \
0  2025-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1  2025-10-25    PROD_002              Adidas Ultraboost 23   Running   
2  2025-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3  2025-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4  2025-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True               26.0   
1  Zapatillas Running          135         True               27.0   
2  Zapatillas Running           85        False                5.0   
3  Zapatillas Running           75        False                3.0   
4        Ropa Running           35        False                3.0   

   precio_venta  ingresos  Amazon  Decathlon  Deporvillage  
0        113.13   2941.38   89.51     113.43        104.78  
1        141.89   

In [17]:
inferencia_df = inferencia_df.drop(columns=['unidades_vendidas'], errors='ignore')

In [19]:
import pandas as pd

ruta_inferencia = "../data/raw/inferencia/ventas_2025_inferencia.csv"

referencia_df = pd.read_csv(ruta_inferencia)

In [20]:
referencia_df['fecha'] = pd.to_datetime(referencia_df['fecha'])

In [21]:
print(referencia_df.shape)

(888, 13)


In [22]:
import holidays

rd_holidays = holidays.DominicanRepublic()

referencia_df['año'] = referencia_df['fecha'].dt.year
referencia_df['mes'] = referencia_df['fecha'].dt.month
referencia_df['dia_mes'] = referencia_df['fecha'].dt.day
referencia_df['dia_semana'] = referencia_df['fecha'].dt.weekday

referencia_df['es_fin_semana'] = referencia_df['dia_semana'].isin([5,6]).astype(int)
referencia_df['es_festivo'] = referencia_df['fecha'].isin(rd_holidays).astype(int)

# Black Friday (último viernes de noviembre)
referencia_df['es_black_friday'] = (
    (referencia_df['mes'] == 11) & 
    (referencia_df['fecha'].dt.weekday == 4) &
    (referencia_df['fecha'].dt.day >= 23)
).astype(int)

# Cyber Monday
referencia_df['es_cyber_monday'] = (
    (referencia_df['mes'] == 11) & 
    (referencia_df['fecha'].dt.weekday == 0) &
    (referencia_df['fecha'].dt.day >= 26)
).astype(int)

In [23]:
# Descuento %
referencia_df['descuento_pct'] = (
    (referencia_df['precio_venta'] - referencia_df['precio_base']) 
    / referencia_df['precio_base']
) * 100

# Precio competencia promedio
referencia_df['precio_competencia'] = referencia_df[
    ['Amazon', 'Decathlon', 'Deporvillage']
].mean(axis=1)

# Ratio precio
referencia_df['ratio_precio'] = (
    referencia_df['precio_venta'] / referencia_df['precio_competencia']
)

# Eliminar columnas competencia
referencia_df = referencia_df.drop(columns=['Amazon', 'Decathlon', 'Deporvillage'])

In [24]:
referencia_df = referencia_df.sort_values(['producto_id', 'fecha'])

for lag in range(1, 8):
    referencia_df[f'lag_{lag}'] = referencia_df.groupby(
        ['producto_id', 'año']
    )['unidades_vendidas'].shift(lag)

# Media móvil 7 días
referencia_df['media_movil_7'] = referencia_df.groupby(
    ['producto_id', 'año']
)['unidades_vendidas'].transform(
    lambda x: x.shift(1).rolling(window=7).mean()
)

In [25]:
# Copias _h
referencia_df['nombre_h'] = referencia_df['nombre']
referencia_df['categoria_h'] = referencia_df['categoria']
referencia_df['subcategoria_h'] = referencia_df['subcategoria']

# One Hot
referencia_df = pd.get_dummies(
    referencia_df,
    columns=['nombre_h', 'categoria_h', 'subcategoria_h']
)

In [26]:
referencia_df = referencia_df.dropna()

In [27]:
referencia_df = referencia_df[referencia_df['mes'] == 11]

In [28]:
referencia_df.to_csv(
    "../data/processed/inferencia_df_transformado.csv",
    index=False
)

In [29]:
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,35.48,106.44,33.84,31.32,34.41
...,...,...,...,...,...,...,...,...,...,...,...,...
883,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,79.64,NaN,88.13,76.22,82.24
884,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,130.00,NaN,137.24,104.62,120.00
885,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,20.18,NaN,19.82,17.71,19.96
886,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,34.79,NaN,33.00,30.41,36.41


In [30]:
inferencia_df.shape


(888, 12)

In [32]:
print(sorted(referencia_df.columns))


['año', 'categoria', 'categoria_h_Fitness', 'categoria_h_Outdoor', 'categoria_h_Running', 'categoria_h_Wellness', 'descuento_pct', 'dia_mes', 'dia_semana', 'es_black_friday', 'es_cyber_monday', 'es_estrella', 'es_festivo', 'es_fin_semana', 'fecha', 'ingresos', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'media_movil_7', 'mes', 'nombre', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900', 'nombre_h_Domyos Kit Mancuernas 20kg', 'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad', 'nombre_h_Lotuscrafts Yoga Bolster', 'nombre_h_Manduka PRO Yoga Mat', 'nombre_h_Merrell Moab 2 GTX', 'nombre_h_New Balance Fresh Foam X 1080v12', 'nombre_h_Nike Air Zoom Pegasus 40', 'nombre_h_Nike Dri-FIT Miler', 'nombre_h_Puma Velocity Nitro 2', 'nombre_h_Quechua MH500', 'nombre_h_Reebok Floatr

In [34]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'precio_venta', 'ingresos', 'Amazon',
       'Decathlon', 'Deporvillage'],
      dtype='str')

In [36]:
df = pd.read_csv("../data/processed/df_final.csv")

In [37]:
print("TRAIN:", len(df.columns))
print("INFERENCIA:", len(referencia_df.columns))

TRAIN: 89
INFERENCIA: 73


In [38]:
features = [
    col for col in referencia_df.columns 
    if col not in ['fecha', 'ingresos', 'unidades_vendidas', 'nombre', 'categoria', 'subcategoria']
]

In [39]:
df = pd.read_csv("../data/processed/df_final.csv")

In [40]:
faltan_en_inferencia = set(df.columns) - set(referencia_df.columns)
sobran_en_inferencia = set(referencia_df.columns) - set(df.columns)

print("FALTAN:", faltan_en_inferencia)
print("SOBRAN:", sobran_en_inferencia)

FALTAN: {'dia', 'dia_anio', 'fin_mes', 'Deporvillage_y', 'nombre_dia', 'Deporvillage_x', 'Decathlon_y', 'Amazon_x', 'Deporvillage', 'fin_trimestre', 'Decathlon_x', 'Decathlon', 'Amazon', 'inicio_trimestre', 'trimestre', 'inicio_mes', 'anio', 'Amazon_y', 'semana_mes', 'semana_anio'}
SOBRAN: {'ratio_precio', 'precio_competencia', 'año', 'dia_mes'}


In [41]:
list(referencia_df.columns)

['fecha',
 'producto_id',
 'nombre',
 'categoria',
 'subcategoria',
 'precio_base',
 'es_estrella',
 'unidades_vendidas',
 'precio_venta',
 'ingresos',
 'año',
 'mes',
 'dia_mes',
 'dia_semana',
 'es_fin_semana',
 'es_festivo',
 'es_black_friday',
 'es_cyber_monday',
 'descuento_pct',
 'precio_competencia',
 'ratio_precio',
 'lag_1',
 'lag_2',
 'lag_3',
 'lag_4',
 'lag_5',
 'lag_6',
 'lag_7',
 'media_movil_7',
 'nombre_h_Adidas Own The Run Jacket',
 'nombre_h_Adidas Ultraboost 23',
 'nombre_h_Asics Gel Nimbus 25',
 'nombre_h_Bowflex SelectTech 552',
 'nombre_h_Columbia Silver Ridge',
 'nombre_h_Decathlon Bandas Elásticas Set',
 'nombre_h_Domyos BM900',
 'nombre_h_Domyos Kit Mancuernas 20kg',
 'nombre_h_Gaiam Premium Yoga Block',
 'nombre_h_Liforme Yoga Pad',
 'nombre_h_Lotuscrafts Yoga Bolster',
 'nombre_h_Manduka PRO Yoga Mat',
 'nombre_h_Merrell Moab 2 GTX',
 'nombre_h_New Balance Fresh Foam X 1080v12',
 'nombre_h_Nike Air Zoom Pegasus 40',
 'nombre_h_Nike Dri-FIT Miler',
 'nombre_h_

In [42]:
list(df.columns)

['fecha',
 'producto_id',
 'nombre',
 'categoria',
 'subcategoria',
 'precio_base',
 'es_estrella',
 'unidades_vendidas',
 'precio_venta',
 'ingresos',
 'Amazon_x',
 'Decathlon_x',
 'Deporvillage_x',
 'anio',
 'dia_semana',
 'es_black_friday',
 'mes',
 'dia',
 'nombre_dia',
 'semana_anio',
 'trimestre',
 'es_fin_semana',
 'es_festivo',
 'es_cyber_monday',
 'inicio_mes',
 'fin_mes',
 'inicio_trimestre',
 'fin_trimestre',
 'dia_anio',
 'semana_mes',
 'lag_1',
 'lag_2',
 'lag_3',
 'lag_4',
 'lag_5',
 'lag_6',
 'lag_7',
 'media_movil_7',
 'descuento_pct',
 'Amazon_y',
 'Decathlon_y',
 'Deporvillage_y',
 'Amazon',
 'Decathlon',
 'Deporvillage',
 'nombre_h_Adidas Own The Run Jacket',
 'nombre_h_Adidas Ultraboost 23',
 'nombre_h_Asics Gel Nimbus 25',
 'nombre_h_Bowflex SelectTech 552',
 'nombre_h_Columbia Silver Ridge',
 'nombre_h_Decathlon Bandas Elásticas Set',
 'nombre_h_Domyos BM900',
 'nombre_h_Domyos Kit Mancuernas 20kg',
 'nombre_h_Gaiam Premium Yoga Block',
 'nombre_h_Liforme Yoga Pad

In [43]:
def crear_features_tiempo(df):
    df = df.copy()

    df['anio'] = df['fecha'].dt.year
    df['mes'] = df['fecha'].dt.month
    df['dia'] = df['fecha'].dt.day
    df['dia_semana'] = df['fecha'].dt.dayofweek

    return df

In [44]:
def procesar_competencia(df):
    df['precio_competencia'] = df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
    return df

In [45]:
def crear_lags(df):
    df = df.sort_values(['producto_id', 'fecha'])

    for i in range(1, 8):
        df[f'lag_{i}'] = df.groupby('producto_id')['unidades_vendidas'].shift(i)

    df['media_movil_7'] = df.groupby('producto_id')['unidades_vendidas'].transform(
        lambda x: x.shift(1).rolling(7).mean()
    )

    return df

In [48]:
def one_hot(df, columnas, encoder=None):
    if encoder is None:
        df = pd.get_dummies(df, columns=columnas)
        return df, df.columns
    else:
        df = pd.get_dummies(df, columns=columnas)
        df = df.reindex(columns=encoder, fill_value=0)
        return df

In [51]:
def crear_features_tiempo(df):
    df = df.copy()

    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

    df['anio'] = df['fecha'].dt.year
    df['mes'] = df['fecha'].dt.month
    df['dia'] = df['fecha'].dt.day
    df['dia_semana'] = df['fecha'].dt.dayofweek

    return df

In [54]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\ojgue\OneDrive\Documentos\FORECASTINGVENTAS\notebooks
['entramiento.ipynb', 'forecasting.ipynb', 'notebooks']


In [55]:
import os

for root, dirs, files in os.walk("."):
    if "features.pkl" in files:
        print("ENCONTRADO EN:", root)

In [58]:
import os

print("DIRECTORIO ACTUAL:")
print(os.getcwd())

print("\nARCHIVOS AQUÍ:")
print(os.listdir())

DIRECTORIO ACTUAL:
c:\Users\ojgue\OneDrive\Documentos\FORECASTINGVENTAS\notebooks

ARCHIVOS AQUÍ:
['entramiento.ipynb', 'forecasting.ipynb', 'notebooks']


In [59]:
import os

for root, dirs, files in os.walk("."):
    if "features.pkl" in files:
        print("ENCONTRADO EN:", os.path.join(root, "features.pkl"))